<a href="https://colab.research.google.com/github/amydiab/analyse-emploi-senegal/blob/diabate/Alternance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
from google.colab import files

# Chargement
uploaded = files.upload()  # sélectionne offres_senegal_AVEC_DESCRIPTIONS.csv
df = pd.read_csv("offres_senegal_AVEC_DESCRIPTIONS.csv")

# Reconversion de la date
mois_fr = {
    "janv.": "Jan", "févr.": "Feb", "mars": "Mar",
    "avr.": "Apr", "mai": "May", "juin": "Jun",
    "juil.": "Jul", "août": "Aug", "sept.": "Sep",
    "oct.": "Oct", "nov.": "Nov", "déc.": "Dec",
    "fév.": "Feb"
}

def convertir_date(date_str):
    if pd.isna(date_str):
        return pd.NaT
    date_str = str(date_str).strip()
    if re.match(r'\d{4}-\d{2}-\d{2}', date_str):
        return pd.to_datetime(date_str, errors="coerce")
    for mois_f, mois_e in mois_fr.items():
        if mois_f in date_str.lower():
            date_str = date_str.lower().replace(mois_f, mois_e)
            return pd.to_datetime(date_str, format="%d %b %Y", errors="coerce")
    return pd.NaT

df["date_publication"] = df["date_publication"].apply(convertir_date)

# Vérification
print(f"✅ Chargé : {df.shape}")
print(f"Colonnes : {df.columns.tolist()}")
print(f"\nValeurs manquantes :")
print(df.isnull().sum())

Saving offres_senegal_AVEC_DESCRIPTIONS.csv to offres_senegal_AVEC_DESCRIPTIONS (1).csv
✅ Chargé : (4215, 11)
Colonnes : ['titre', 'ville', 'date_publication', 'lien', 'source', 'contrat', 'entreprise', 'secteur', 'secteur_categorie', 'mois', 'description']

Valeurs manquantes :
titre                  0
ville                 79
date_publication     120
lien                   0
source                 0
contrat              554
entreprise           228
secteur              228
secteur_categorie      0
mois                 120
description            0
dtype: int64


In [ ]:
import re

def extraire_experience(description):
    if pd.isna(description):
        return None

    texte = str(description).lower()

    # Patterns pour détecter les années d'expérience
    patterns = [
        r'(\d+)\s*ans?\s*d[\'e]\s*exp[eé]rience',
        r'exp[eé]rience\s*[:\-]?\s*(\d+)\s*ans?',
        r'minimum\s*(\d+)\s*ans?',
        r'au\s*moins\s*(\d+)\s*ans?',
        r'(\d+)\s*to\s*(\d+)\s*years?',
        r'(\d+)\s*years?\s*of\s*experience',
        r'exp[eé]rience\s*de\s*(\d+)',
        r'(\d+)\s*[-à]\s*(\d+)\s*ans?\s*d[\'e]\s*exp[eé]rience',
    ]

    for pattern in patterns:
        match = re.search(pattern, texte)
        if match:
            # Retourne le premier nombre trouvé
            return int(match.group(1))

    # Patterns qualitatifs
    if any(mot in texte for mot in ["débutant", "junior", "sans expérience", "jeune diplômé"]):
        return 0
    if any(mot in texte for mot in ["senior", "confirmé", "expérimenté"]):
        return 5

    return None

df["experience_annees"] = df["description"].apply(extraire_experience)

print("=== Expériences extraites ===")
print(f"Trouvées : {df['experience_annees'].notna().sum()} / {len(df)}")
print(f"\nRépartition :")
print(df["experience_annees"].value_counts().sort_index().head(15))
print(f"\nMoyenne : {df['experience_annees'].mean():.1f} ans")

=== Expériences extraites ===
Trouvées : 1675 / 4215

Répartition :
experience_annees
0.0     207
1.0      50
2.0     275
3.0     207
4.0      24
5.0     628
6.0       8
7.0     129
8.0      19
9.0       2
10.0     50
12.0      4
15.0      7
18.0      6
20.0     57
Name: count, dtype: int64

Moyenne : 4.5 ans


In [ ]:
# Vérifications des cas suspects entre 20 ans et 18 ans
print("=== Offres avec 20 ans d'expérience ===")
offres_20 = df[df["experience_annees"] == 20][["titre", "description"]].head(3)
for _, row in offres_20.iterrows():
    print(f"\nTitre : {row['titre']}")
# Chercher le contexte autour du pattern
    texte = str(row['description']).lower()
    idx = texte.find("20")
    print(f"Contexte : ...{texte[max(0,idx-50):idx+80]}...")

=== Offres avec 20 ans d'expérience ===

Titre : Enseignants
Contexte : ...e au secondaire (6e à 12e année) posté le 10 nov. 2024 iqra bilingual academy formations, éducation, enseignement supérieur - univ...

Titre : Regional Industry Director, Infrastructure & Natural Resources
Contexte : ...e au secondaire (6e à 12e année) posté le 10 nov. 2024 iqra bilingual academy formations, éducation, enseignement supérieur - univ...

Titre : Technicien(ne) en Génie Électrique
Contexte : ...ite de dépôt des candidatures vendredi 23 janvier 2026 à 00:00 postuler limite de dépôt des candidatures vendredi 23 janvier 2026 ...


In [ ]:
def extraire_experience_v2(description):
    if pd.isna(description):
        return None

    texte = str(description).lower()

    # Patterns corrigés — on exclut les nombres > 15
    # et on s'assure que le contexte est bien "expérience"
    patterns = [
        r'(\d+)\s*ans?\s*d[\'e]\s*exp[eé]rience',
        r'exp[eé]rience\s*[:\-]?\s*(\d+)\s*ans?',
        r'minimum\s*(\d+)\s*ans?\s*d[\'e]\s*exp[eé]rience',
        r'au\s*moins\s*(\d+)\s*ans?\s*d[\'e]\s*exp[eé]rience',
        r'(\d+)\s*years?\s*of\s*experience',
        r'exp[eé]rience\s*de\s*(\d+)\s*ans?',
        r'(\d+)\s*[-à]\s*(\d+)\s*ans?\s*d[\'e]\s*exp[eé]rience',
        r'exp[eé]rience\s*:\s*(\d+)',
    ]

    for pattern in patterns:
        match = re.search(pattern, texte)
        if match:
            valeur = int(match.group(1))
            # On exclut les valeurs > 15 — clairement des années calendaires
            if valeur <= 15:
                return valeur

    # Patterns qualitatifs
    if any(mot in texte for mot in ["débutant", "junior", "sans expérience", "jeune diplômé"]):
        return 0
    if "senior" in texte or "confirmé" in texte or "expérimenté" in texte:
        return 5

    return None

df["experience_annees"] = df["description"].apply(extraire_experience_v2)

print("=== Expériences corrigées ===")
print(f"Trouvées : {df['experience_annees'].notna().sum()} / {len(df)}")
print(f"\nRépartition :")
print(df["experience_annees"].value_counts().sort_index())
print(f"\nMoyenne : {df['experience_annees'].mean():.1f} ans")

=== Expériences corrigées ===
Trouvées : 1071 / 4215

Répartition :
experience_annees
0.0     276
1.0      19
2.0      67
3.0      77
4.0       6
5.0     592
6.0       4
7.0       4
8.0      13
9.0       1
10.0     10
12.0      2
Name: count, dtype: int64

Moyenne : 3.4 ans


In [ ]:
import pandas as pd
import re
from google.colab import files

# Chargement
uploaded = files.upload()  # sélectionne offres_senegal_AVEC_DESCRIPTIONS.csv
df = pd.read_csv("offres_senegal_AVEC_DESC-Exp - offres_senegal_AVEC_DESCRIPTIONS.csv")

# Reconversion de la date
mois_fr = {
    "janv.": "Jan", "févr.": "Feb", "mars": "Mar",
    "avr.": "Apr", "mai": "May", "juin": "Jun",
    "juil.": "Jul", "août": "Aug", "sept.": "Sep",
    "oct.": "Oct", "nov.": "Nov", "déc.": "Dec",
    "fév.": "Feb"
}

def convertir_date(date_str):
    if pd.isna(date_str):
        return pd.NaT
    date_str = str(date_str).strip()
    if re.match(r'\d{4}-\d{2}-\d{2}', date_str):
        return pd.to_datetime(date_str, errors="coerce")
    for mois_f, mois_e in mois_fr.items():
        if mois_f in date_str.lower():
            date_str = date_str.lower().replace(mois_f, mois_e)
            return pd.to_datetime(date_str, format="%d %b %Y", errors="coerce")
    return pd.NaT

df["date_publication"] = df["date_publication"].apply(convertir_date)

# Vérification
print(f"✅ Chargé : {df.shape}")
print(f"Colonnes : {df.columns.tolist()}")
print(f"\nValeurs manquantes :")
print(df.isnull().sum())

Saving offres_senegal_AVEC_DESC-Exp - offres_senegal_AVEC_DESCRIPTIONS.csv to offres_senegal_AVEC_DESC-Exp - offres_senegal_AVEC_DESCRIPTIONS (1).csv
✅ Chargé : (4215, 12)
Colonnes : ['titre', 'ville', 'date_publication', 'lien', 'source', 'contrat', 'entreprise', 'secteur', 'secteur_categorie', 'mois', 'Expérience', 'description']

Valeurs manquantes :
titre                   0
ville                  79
date_publication      120
lien                    0
source                  0
contrat               554
entreprise            228
secteur               228
secteur_categorie       0
mois                  120
Expérience           2648
description             0
dtype: int64


In [ ]:
print("=== Contenu colonne Expérience ===")
print(df["Expérience"].value_counts().head(30).to_string())
print(f"\nValeurs uniques : {df['Expérience'].nunique()}")

=== Contenu colonne Expérience ===
Expérience
2 ans          450
5 ans          302
3 ans          294
10 ans          81
3 à 5 ans       60
2 à 3 ans       46
4 ans           42
7 ans           40
8 ans           25
1 à 2 ans       21
5 à 10 ans      20
18 ans          17
5 à 7 ans       16
15 ans          14
5 à 8 ans       12
2 à 5 ans       12
6 ans           11
30 ans          11
1 à 3 ans       11
2 à 4 ans        7
12 ans           6
8 à 10 ans       5
05 ans           5
21 ans           4
10 à 15 ans      4
10+ ans          4
4 à 7 ans        3
7 à 10 ans       3
03 ans           3
3 à 7 ans        3

Valeurs uniques : 54


In [ ]:
# Vérification des valeurs suspectes
for val in ["30 ans", "21 ans", "18 ans"]:
    print(f"\n=== {val} ===")
    exemples = df[df["Expérience"] == val][["titre", "description"]].head(2)
    for _, row in exemples.iterrows():
        print(f"Titre : {row['titre']}")
        texte = str(row['description']).lower()
        idx = texte.find(val.replace(" ans", ""))
        print(f"Contexte : ...{texte[max(0,idx-60):idx+80]}...")
        print()



=== 30 ans ===
Titre : Nounou et Chef Cuisinière Expérimentées
Contexte : ...nement propre et organisé. profil recherché •	femme âgée de 30 ans minimum. •	expérience confirmée dans la garde d’enfants. •	patiente, séri...

Titre : Responsable Ressources Humaines
Contexte : ...treaming. en afrique, le groupe est implanté depuis plus de 30 ans et opère dans plus de 20 pays. dans ce cadre, canal+ sénégal recherche so...


=== 21 ans ===
Titre : Technicien en Logistique
Contexte : ...sidence casier judiciaire de – 3 mois profil âge minimum de 21 ans maximum 56 ans diplôme néant apte physiquement savoir lire écrire et parl...

Titre : Agent de sécurité
Contexte : ...aire datant de moins de 3 mois. profil requis âge : minimum 21 ans, maximum 56 ans. diplôme : non requis. apte physiquement. savoir lire, éc...


=== 18 ans ===
Titre : Agent de sécurité incendie
Contexte : ...ses droits civiques et être de bonne moralité ; être âgé de 18 ans au moins et de 40 ans au plus. être titulaire du bacc

In [ ]:
# Nettoyage de la colonne Expérience
def nettoyer_experience(exp):
    if pd.isna(exp):
        return None

    exp = str(exp).strip()

    # Supprime les zéros devant : 05 ans → 5 ans, 03 ans → 3 ans
    exp = re.sub(r'^0(\d)', r'\1', exp)

    # Valeurs aberrantes — ages ou années calendaires
    valeurs_aberrantes = ["30 ans", "21 ans", "18 ans"]
    if exp in valeurs_aberrantes:
        return None

    return exp

df["Expérience"] = df["Expérience"].apply(nettoyer_experience)

print("=== Expérience nettoyée ===")
print(df["Expérience"].value_counts().to_string())
print(f"\nManquants : {df['Expérience'].isna().sum()}")
print(f"\nValeurs uniques : {df['Expérience'].nunique()}")

=== Expérience nettoyée ===
Expérience
2 ans          450
5 ans          308
3 ans          297
10 ans          81
3 à 5 ans       61
2 à 3 ans       46
4 ans           42
7 ans           40
8 ans           25
1 à 2 ans       21
5 à 10 ans      20
5 à 7 ans       16
15 ans          14
2 à 5 ans       13
5 à 8 ans       12
6 ans           11
1 à 3 ans       11
2 à 4 ans        7
12 ans           6
8 à 10 ans       5
10+ ans          4
10 à 15 ans      4
7 à 10 ans       3
4 à 7 ans        3
0 à 2 ans        3
3 à 7 ans        3
1 an             3
5 ans            2
5+ ans           2
9 ans            2
0 à 8 ans        2
5 à 6 ans        2
6 à 8 ans        2
0 à 03 ans       2
2   ans          1
0 à 3 ans        1
1 ans            1
10 ans           1
0 à 5 ans        1
2 à 10 ans       1
1 à 2ans         1
3 à 4 ans        1
3 à 6 ans        1
12ans            1
4 à 5 ans        1
3 à 05 ans       1

Manquants : 2680

Valeurs uniques : 46


In [ ]:
def clean_experience(exp):
    if pd.isna(exp):
        return None

    exp = str(exp).lower().strip()

    # enlever espaces bizarres
    exp = re.sub(r"\s+", " ", exp)

    # uniformiser "ans"
    exp = exp.replace("ans", "").strip()

    # corriger formats type "12ans"
    exp = re.sub(r"(\d)([a-z])", r"\1 \2", exp)

    # enlever zéros inutiles
    exp = re.sub(r'\b0+(\d)', r'\1', exp)

    return exp

In [ ]:
df["exp_clean"] = df["Expérience"].apply(clean_experience)

In [ ]:
def standardize_experience(exp):
    if exp is None:
        return None

    # cas intervalle
    match_range = re.findall(r"\d+", exp)

    if len(match_range) == 2:
        return f"{match_range[0]}-{match_range[1]}"

    # cas "10+"
    if "+" in exp:
        num = re.findall(r"\d+", exp)
        if num:
            return f"{num[0]}+"

    # cas simple
    if len(match_range) == 1:
        return match_range[0]

    return None

In [ ]:
df["experience_final"] = df["exp_clean"].apply(standardize_experience)

In [ ]:
df["experience_final"].value_counts().head(20)

,count
experience_final,
2,451
5,310
3,297
10,82
3-5,62
2-3,46
4,42
7,40
8,25


In [ ]:
from collections import Counter
import re
import pandas as pd

# Define the cleaning function for description text
def clean_description_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    # Remove special characters and punctuation, keep alphanumeric and spaces
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # Replace multiple spaces with a single space and strip leading/trailing spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Create the 'desc_clean' column
df["desc_clean"] = df["description"].apply(clean_description_text)

words = " ".join(df["desc_clean"]).split()

word_counts = Counter(words)

word_counts.most_common(50)

[('de', 121799),
 ('et', 105785),
 ('des', 76738),
 ('les', 55439),
 ('la', 53587),
 ('en', 38051),
 ('du', 34220),
 ('le', 32644),
 ('pour', 18840),
 ('dans', 16915),
 ('sur', 15572),
 ('un', 15210),
 ('une', 13238),
 ('avec', 12787),
 ('poste', 12785),
 ('au', 11118),
 ('and', 11048),
 ('ou', 10357),
 ('gestion', 9828),
 ('aux', 9471),
 ('description', 9268),
 ('partager', 8466),
 ('lannonce', 8439),
 ('dakar', 7929),
 ('exprience', 6365),
 ('postuler', 6059),
 ('assurer', 5995),
 ('clients', 5937),
 ('ans', 5864),
 ('est', 5823),
 ('vous', 5654),
 ('capacit', 5361),
 ('suivi', 5286),
 ('travail', 5178),
 ('communication', 5131),
 ('comptences', 5111),
 ('projet', 5051),
 ('que', 4978),
 ('par', 4927),
 ('matrise', 4732),
 ('projets', 4671),
 ('2025', 4616),
 ('to', 4295),
 ('the', 4258),
 ('dveloppement', 4111),
 ('post', 4085),
 ('nous', 4027),
 ('profil', 3845),
 ('temps', 3825),
 ('bonne', 3803)]

In [ ]:
from collections import Counter
from itertools import islice

words = " ".join(df["desc_clean"]).split()

# créer des paires de mots
bigrams = list(zip(words, words[1:]))

bigram_counts = Counter(bigrams)

bigram_counts.most_common(30)

[(('de', 'la'), 15409),
 (('du', 'poste'), 9734),
 (('et', 'de'), 8650),
 (('description', 'du'), 8473),
 (('partager', 'lannonce'), 8320),
 (('lannonce', 'sur'), 6623),
 (('et', 'les'), 6056),
 (('et', 'des'), 5262),
 (('dans', 'le'), 5106),
 (('avec', 'les'), 4724),
 (('et', 'la'), 4614),
 (('post', 'le'), 4048),
 (('dakar', 'description'), 3667),
 (('sur', 'gao'), 3339),
 (('gao', 'partager'), 3330),
 (('la', 'gestion'), 3154),
 (('gestion', 'des'), 3152),
 (('ans', 'dexprience'), 3011),
 (('de', 'travail'), 2867),
 (('dans', 'la'), 2864),
 (('mise', 'en'), 2825),
 (('de', 'lentreprise'), 2803),
 (('dans', 'les'), 2775),
 (('en', 'uvre'), 2686),
 (('sur', 'les'), 2676),
 (('gestion', 'de'), 2660),
 (('temps', 'complet'), 2649),
 (('complet', 'sans'), 2581),
 (('sans', 'tltravail'), 2580),
 (('profil', 'recherch'), 2489)]

In [ ]:
stop_words = set([
    "de", "et", "la", "le", "les", "des", "du", "en",
    "à", "un", "une", "avec", "pour", "dans",
    "au", "aux", "sur", "ou", "que",
    "the", "and", "of", "to", "in",
    "l", "d"
])

In [ ]:
words = [
    w for w in " ".join(df["desc_clean"]).split()
    if w not in stop_words and len(w) > 3
]

In [ ]:
bigrams = list(zip(words, words[1:]))

from collections import Counter
bigram_counts = Counter(bigrams)

bigram_counts.most_common(30)

[(('description', 'poste'), 8478),
 (('partager', 'lannonce'), 8320),
 (('lannonce', 'partager'), 5014),
 (('dakar', 'description'), 3681),
 (('temps', 'complet'), 2649),
 (('complet', 'sans'), 2581),
 (('sans', 'tltravail'), 2580),
 (('profil', 'recherch'), 2489),
 (('postuler', 'postuler'), 2377),
 (('limite', 'candidatures'), 1958),
 (('mise', 'uvre'), 1739),
 (('postuler', 'ajouter'), 1714),
 (('ajouter', 'favoris'), 1702),
 (('favoris', 'partager'), 1697),
 (('0000', 'postuler'), 1668),
 (('lannonce', 'xtwitter'), 1644),
 (('xtwitter', 'partager'), 1639),
 (('lannonce', 'whatsapp'), 1621),
 (('whatsapp', 'propos'), 1611),
 (('propos', 'lentreprise'), 1604),
 (('tltravail', 'dexprience'), 1592),
 (('vous', 'intresser'), 1526),
 (('offres', 'pourraient'), 1452),
 (('pourraient', 'galement'), 1446),
 (('galement', 'vous'), 1445),
 (('prcdent', 'suivant'), 1434),
 (('intresser', 'prcdent'), 1432),
 (('capacit', 'travailler'), 1354),
 (('missions', 'principales'), 1333),
 (('2025', '00

In [ ]:
noise_words = [
    "description", "poste", "annonce", "partager",
    "twitter", "whatsapp", "favoris", "postuler",
    "ajouter", "offres", "suivant", "précédent",
    "dakar", "2025"
]

In [ ]:
df = df.copy()

In [ ]:
def remove_noise(text):
    for word in noise_words:
        text = text.replace(word, "")
    return text

df["desc_clean2"] = df["desc_clean"].apply(remove_noise)

In [ ]:
words = [
    w for w in " ".join(df["desc_clean2"]).split()
    if w not in stop_words and len(w) > 3
]

bigrams = list(zip(words, words[1:]))

from collections import Counter
bigram_counts = Counter(bigrams)

bigram_counts.most_common(30)

[(('temps', 'complet'), 2649),
 (('complet', 'sans'), 2581),
 (('sans', 'tltravail'), 2580),
 (('profil', 'recherch'), 2489),
 (('limite', 'candidatures'), 1958),
 (('mise', 'uvre'), 1739),
 (('propos', 'lentreprise'), 1604),
 (('tltravail', 'dexprience'), 1592),
 (('vous', 'intresser'), 1526),
 (('pourraient', 'galement'), 1446),
 (('galement', 'vous'), 1445),
 (('intresser', 'prcdent'), 1432),
 (('missions', 'principales'), 1421),
 (('capacit', 'travailler'), 1354),
 (('0000', 'limite'), 1127),
 (('rseaux', 'sociaux'), 1117),
 (('matrise', 'outils'), 1083),
 (('lettre', 'motivation'), 1026),
 (('bonne', 'connaissance'), 912),
 (('post', 'juil'), 888),
 (('assurer', 'suivi'), 870),
 (('post', 'janv'), 844),
 (('date', 'limite'), 804),
 (('juil', '2024'), 794),
 (('bonne', 'matrise'), 781),
 (('nous', 'recherchons'), 768),
 (('formations', 'ducation'), 759),
 (('sous', 'supervision'), 737),
 (('mettre', 'uvre'), 735),
 (('exprience', 'professionnelle'), 699)]

In [ ]:
words = [
    w for w in " ".join(df["desc_clean2"]).split()
    if w not in stop_words
    and len(w) > 3
    and not w.isdigit()
]

In [ ]:
from collections import Counter

word_counts = Counter(words)

word_counts.most_common(50)

[('gestion', 9829),
 ('exprience', 6365),
 ('assurer', 5995),
 ('clients', 5937),
 ('vous', 5655),
 ('capacit', 5361),
 ('suivi', 5286),
 ('travail', 5178),
 ('communication', 5131),
 ('comptences', 5111),
 ('projet', 5051),
 ('matrise', 4732),
 ('projets', 4671),
 ('dveloppement', 4111),
 ('post', 4085),
 ('nous', 4027),
 ('profil', 3845),
 ('temps', 3825),
 ('bonne', 3803),
 ('grer', 3752),
 ('formation', 3574),
 ('mise', 3547),
 ('outils', 3457),
 ('missions', 3412),
 ('sens', 3396),
 ('dexprience', 3389),
 ('donnes', 3368),
 ('sngal', 3358),
 ('activits', 3303),
 ('sans', 3264),
 ('votre', 3190),
 ('lentreprise', 3183),
 ('connaissance', 3085),
 ('scurit', 3059),
 ('responsable', 2926),
 ('candidatures', 2922),
 ('marketing', 2884),
 ('qualit', 2838),
 ('travailler', 2832),
 ('complet', 2809),
 ('mission', 2761),
 ('services', 2730),
 ('principales', 2716),
 ('uvre', 2707),
 ('service', 2692),
 ('tltravail', 2677),
 ('techniques', 2661),
 ('quipes', 2619),
 ('participer', 2608),
 (

In [ ]:
df[["description", "desc_clean", "desc_clean2"]].head(3)

,description,desc_clean,desc_clean2
0,"Description de l'offre A propos de nous : SamaMbey SUARL est une entreprise sociale évoluant dans le domaine agricole.Nous intervenons dans de nombreuses régions du Sénégal. Sama Mbey vise à améliorer les conditions de vie des petits producteurs en milieu rural et veut les aider à sortir de la pauvreté. Chez SamaMbey les petits agriculteurs peuvent se financer s'ils ont le bon outil pour payer petit à petit en avance pour les intrants. Notre vision et objectif à travers notre Étoile polaire sont de travailler avec 1 million de petits producteurs à l’horizon 2026. En plus, SamaMbey Suarl s’engage dans le domaine du développement communautaire en menant des actions de sensibilisation en nutrition, santé et hygiène. Intitulé du poste : Agent distributeur Localisation : La prestation se déroulera en milieu rural, dans les zones d'intervention de Sama Mbey, réparties sur trois régions administratives du Sénégal : Matam Bakel, Hamady Ounare, Matam, Thilogne Kolda Dabo, Bourocco,Kounkane, Médina Yoro Foula, Tanaff, Vélingara Sédhiou / Ziguinchor Bignona, Bounkiling, Diattacounda, Marsassoum, Médina Wandifa Nombre de postes: 36 Type de contrat: Prestation de service Rôles et responsabilités Les agents distributeurs auront pour missions : 1. VENTE & MOBILISATION - Développement commercial - Assister l'équipe de vente dans l'atteinte des objectifs de paquets finis - Réviser l'outil « Constitution de paquet 2026 » pour le maîtriser - Organiser les intrants selon l'outil de décharge",description de loffre a propos de nous samambey suarl est une entreprise sociale voluant dans le domaine agricolenous intervenons dans de nombreuses rgions du sngal sama mbey vise amliorer les conditions de vie des petits producteurs en milieu rural et veut les aider sortir de la pauvret chez samambey les petits agriculteurs peuvent se financer sils ont le bon outil pour payer petit petit en avance pour les intrants notre vision et objectif travers notre toile polaire sont de travailler avec 1 million de petits producteurs lhorizon 2026 en plus samambey suarl sengage dans le domaine du dveloppement communautaire en menant des actions de sensibilisation en nutrition sant et hygine intitul du poste agent distributeur localisation la prestation se droulera en milieu rural dans les zones dintervention de sama mbey rparties sur trois rgions administratives du sngal matam bakel hamady ounare matam thilogne kolda dabo bouroccokounkane mdina yoro foula tanaff vlingara sdhiou ziguinchor bignona bounkiling diattacounda marsassoum mdina wandifa nombre de postes 36 type de contrat prestation de service rles et responsabilits les agents distributeurs auront pour missions 1 vente mobilisation dveloppement commercial assister lquipe de vente dans latteinte des objectifs de paquets finis rviser loutil constitution de paquet 2026 pour le matriser organiser les intrants selon loutil de dcharge,de loffre a propos de nous samambey suarl est une entreprise sociale voluant dans le domaine agricolenous intervenons dans de nombreuses rgions du sngal sama mbey vise amliorer les conditions de vie des petits producteurs en milieu rural et veut les aider sortir de la pauvret chez samambey les petits agriculteurs peuvent se financer sils ont le bon outil pour payer petit petit en avance pour les intrants notre vision et objectif travers notre toile polaire sont de travailler avec 1 million de petits producteurs lhorizon 2026 en plus samambey suarl sengage dans le domaine du dveloppement communautaire en menant des actions de sensibilisation en nutrition sant et hygine intitul du agent distributeur localisation la prestation se droulera en milieu rural dans les zones dintervention de sama mbey rparties sur trois rgions administratives du sngal matam bakel hamady ounare matam thilogne kolda dabo bouroccokounkane mdina yoro foula tanaff vlingara sdhiou ziguinchor bignona bounkiling diattacounda marsassoum mdina wandifa nombre de s 36 type de contrat pr

In [ ]:
useful_words = [
    w for w, freq in word_counts.most_common(100)
    if w not in stop_words
    and w not in noise_words
    and w not in ["vous", "nous", "votre", "etc", "sans"]
    and len(w) > 4
]

In [ ]:
useful_words[:30]

['gestion',
 'exprience',
 'assurer',
 'clients',
 'capacit',
 'suivi',
 'travail',
 'communication',
 'comptences',
 'projet',
 'matrise',
 'projets',
 'dveloppement',
 'profil',
 'temps',
 'bonne',
 'formation',
 'outils',
 'missions',
 'dexprience',
 'donnes',
 'sngal',
 'activits',
 'lentreprise',
 'connaissance',
 'scurit',
 'responsable',
 'candidatures',
 'marketing',
 'qualit']

In [ ]:
def extract_context(text, keyword, window=3):
    words = text.split()
    contexts = []

    for i, word in enumerate(words):
        if word == keyword:
            start = max(i - window, 0)
            end = min(i + window + 1, len(words))
            context = " ".join(words[start:end])
            contexts.append(context)

    return contexts

In [ ]:
contexts = []

for text in df["desc_clean2"].head(200):  # on teste sur un échantillon
    contexts.extend(extract_context(text, "données"))

contexts[:20]

[]

In [ ]:
contexts = []

for text in df["desc_clean2"].head(200):  # on teste sur un échantillon
    contexts.extend(extract_context(text, "développement"))

contexts[:20]

[]

In [ ]:
keywords = ["données", "développement", "projet", "marketing", "sécurité"]

In [ ]:
def extract_skills_from_context(text, keywords, window=3):
    words = text.split()
    skills = []

    for i, word in enumerate(words):
        if word in keywords:
            start = max(i - window, 0)
            end = min(i + window + 1, len(words))

            context = words[start:end]

            # garder uniquement 2-3 mots autour
            skill = " ".join(context)

            skills.append(skill)

    return skills

In [ ]:
df["skills_raw"] = df["desc_clean2"].apply(
    lambda x: extract_skills_from_context(x, keywords)
)

In [ ]:
def clean_skills(skills):
    cleaned = []

    for s in skills:
        if "données" in s:
            if "analyse" in s:
                cleaned.append("analyse de données")
            elif "collecte" in s:
                cleaned.append("collecte de données")
            elif "traitement" in s:
                cleaned.append("traitement de données")
            elif "visualisation" in s:
                cleaned.append("visualisation de données")

    return list(set(cleaned))

In [ ]:
df["skills_final"] = df["skills_raw"].apply(clean_skills)

In [ ]:
def clean_dev_skills(contexts):
    skills = []

    for s in contexts:
        if "développement" in s:

            if "application" in s:
                skills.append("développement d'applications")

            elif "api" in s:
                skills.append("développement d'api")

            elif "interface" in s:
                skills.append("développement d'interfaces")

            elif "commercial" in s:
                skills.append("développement commercial")

    return list(set(skills))

In [ ]:
df["dev_skills"] = df["skills_raw"].apply(clean_dev_skills)

In [ ]:
contexts = []

for text in df["desc_clean2"].head(200):  # on teste sur un échantillon
    contexts.extend(extract_context(text, "projet"))

contexts[:20]

['le cadre du projet dfinir les priorits',
 'et objectifs du projet et laborer un',
 'coordonner lexcution du projet en troite collaboration',
 'la dynamique du projet votre profil bac',
 'en uvre le projet de rintgration des',
 'rapatris rrp8 le projet soutient galement lintgration',
 'villages bnficiaires du projet le comit dvaluation',
 'manager chefs de projet communication vnementiel chargs',
 'vnementiel chargs de projet cration contenus responsabilits',
 'efficaces gestion de projet validation production standardiser',
 'en uvre du projet dcrit en annexe',
 'enjeux cls du projet ce documentaire devra',
 'et rsultats du projet renforcer le plaidoyer',
 'de plaidoyer du projet objectifs spcifiques concevoir',
 'le document de projet raliser des prises',
 'du document de projet laboration du c',
 'marketing gestion de projet s temps plein',
 'encaiss sur chaque projet livr site vitrine',
 'lquipe clinical du projet epic de fhi360',
 'directrice technique du projet ilelle appuiera l

In [ ]:
contexts = []

for text in df["desc_clean2"].head(200):  # on teste sur un échantillon
    contexts.extend(extract_context(text, "marketing"))

contexts[:20]

['en commerce international marketing exprience professionnelle souhaite',
 'finances ventes et marketing gestion de projet',
 'et aux campagnes marketing contribuer lorganisation et',
 'formation en commerce marketing ou gestion exprience',
 'en communication et marketing digital pour continuer',
 'formation en communication marketing ou quivalent excellentes',
 'required with communications marketing or public relations',
 'bac5 en commerce marketing communication ou quivalent',
 'une charge trade marketing sous la responsabilit',
 'intrt pour le marketing digital et la',
 'lquipe dexploitation actions marketing participer la mise',
 'diplme en commerce marketing agroalimentaire ou nutrition',
 'communication et du marketing social crant ainsi']

In [ ]:
contexts = []

for text in df["desc_clean2"].head(200):  # on teste sur un échantillon
    contexts.extend(extract_context(text, "banque"))

contexts[:20]

['etc organiser une banque dimages et de',
 'engages par la banque centrale concevoir et',
 'bac 5 en banque finance conomie gestion',
 'des utilisateurs en banque institution publique ou',
 'efficient de la banque localisation sige de',
 'services de la banque ou formuler toute',
 'externe de la banque centrale en cohrence',
 'messages de la banque centrale localisation sige',
 'sociaux de la banque participe la veille',
 'perception de la banque centrale dans lespace',
 'obligatoire en financement banque microfinance cabinet ou',
 'directives de la banque mondiale et du']

In [ ]:
contexts = []

for text in df["desc_clean2"].head(200):  # on teste sur un échantillon
    contexts.extend(extract_context(text, "energies"))

contexts[:20]

['de loffre vinci energies building solutions marque',
 'marque de vinci energies renforce ses quipes',
 'offre cdi vinci energies sngal recrute un',
 'plomberie hf vinci energies building solutions marque',
 'marque de vinci energies renforce ses quipes']

In [ ]:
categories = {
    "data": ["données", "analyse", "traitement", "visualisation"],
    "dev": ["développement", "application", "api", "interface"],
    "marketing": ["marketing", "communication", "réseaux"],
    "finance": ["finance", "comptabilité", "banque"],
    "projet": ["projet", "gestion", "suivi"],
    "agriculture": ["agriculture", "rural", "agronomie"],
    "administration": ["administratif", "gestion administrative"]
}

In [ ]:
def categorize_job(text):
    found_categories = []

    for category, keywords in categories.items():
        for kw in keywords:
            if kw in text:
                found_categories.append(category)
                break

    return list(set(found_categories))

In [ ]:
df["job_category"] = df["desc_clean2"].apply(categorize_job)

In [ ]:
df["job_category_str"] = df["job_category"].apply(lambda x: ", ".join(x))

In [ ]:
df["secteur"].value_counts().head(50)

,count
secteur,
"Emploi, Agences de recrutement",506
"Administrations, Organisations non-gouvernementales (ONG)",463
"Administrations, Organismes internationaux",216
"Commerces, Vente en ligne",195
"Comptabilité, juridique et conseil, Gestion des ressources humaines",181
"Emploi, Travail temporaire - Intérim",124
"Formations, éducation, Enseignement supérieur - Université",88
"Finances, Banques",87
"Comptabilité, juridique et conseil, Management",66


In [ ]:
def clean_secteur(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    # normalisation simple
    text = text.replace("&", " ")
    text = text.replace("/", " ")
    text = text.replace("-", " ")

    return text

df["secteur_clean"] = df["secteur"].apply(clean_secteur)

In [ ]:
from collections import Counter

sect_words = " ".join(df["secteur_clean"]).split()

sect_counts = Counter(sect_words)

sect_counts.most_common(30)

[('de', 875),
 ('administrations,', 788),
 ('agences', 655),
 ('et', 653),
 ('emploi,', 637),
 ('recrutement', 506),
 ('organisations', 463),
 ('non', 463),
 ('gouvernementales', 463),
 ('(ong)', 463),
 ('juridique', 365),
 ('comptabilité,', 359),
 ('conseil,', 359),
 ('commerces,', 238),
 ('organismes', 218),
 ('internationaux', 216),
 ('finances,', 206),
 ('vente', 200),
 ('en', 196),
 ('ligne', 195),
 ('gestion', 182),
 ('formations,', 182),
 ('éducation,', 182),
 ('des', 181),
 ('ressources', 181),
 ('humaines', 181),
 ('communication,', 174),
 ('publicité,', 174),
 ('agroalimentaire,', 133),
 ('travail', 124)]

In [ ]:
categories = {
    "finance": ["finance", "banque"],
    "marketing": ["marketing", "communication"],
    "tech": ["informatique", "it", "digital"],
    "agriculture": ["agriculture", "agronomie"],
    "logistique": ["logistique", "transport"],
    "sante": ["santé", "medical"],
    "education": ["education", "formation"],
    "administration": ["administration", "public"]
}

In [ ]:
def categorize_from_secteur(text):
    found = []

    for cat, keywords in categories.items():
        for kw in keywords:
            if kw in text:
                found.append(cat)
                break

    return list(set(found))

In [ ]:
df["sector_category_final"] = df["secteur_clean"].apply(categorize_from_secteur)

In [ ]:
df["sector_category_str"] = df["sector_category_final"].apply(lambda x: ", ".join(x))

In [ ]:
sect_counts.most_common(30)

[('de', 875),
 ('administrations,', 788),
 ('agences', 655),
 ('et', 653),
 ('emploi,', 637),
 ('recrutement', 506),
 ('organisations', 463),
 ('non', 463),
 ('gouvernementales', 463),
 ('(ong)', 463),
 ('juridique', 365),
 ('comptabilité,', 359),
 ('conseil,', 359),
 ('commerces,', 238),
 ('organismes', 218),
 ('internationaux', 216),
 ('finances,', 206),
 ('vente', 200),
 ('en', 196),
 ('ligne', 195),
 ('gestion', 182),
 ('formations,', 182),
 ('éducation,', 182),
 ('des', 181),
 ('ressources', 181),
 ('humaines', 181),
 ('communication,', 174),
 ('publicité,', 174),
 ('agroalimentaire,', 133),
 ('travail', 124)]

In [ ]:
def split_secteurs(text):
    if pd.isna(text):
        return []

    # on met tout en minuscule
    text = text.lower()

    # on split par virgule
    parts = [p.strip() for p in text.split(",")]

    return parts

df["secteur_list"] = df["secteur"].apply(split_secteurs)

In [ ]:
all_secteurs = [item for sublist in df["secteur_list"] for item in sublist]

In [ ]:
from collections import Counter

sect_counts = Counter(all_secteurs)

sect_counts.most_common(30)

[('administrations', 874),
 ('emploi', 649),
 ('agences de recrutement', 506),
 ('organisations non-gouvernementales (ong)', 463),
 ('comptabilité', 359),
 ('juridique et conseil', 359),
 ('commerces', 238),
 ('organismes internationaux', 216),
 ('finances', 206),
 ('vente en ligne', 195),
 ('formations', 182),
 ('éducation', 182),
 ('gestion des ressources humaines', 181),
 ('publicité', 179),
 ('communication', 174),
 ('industries', 173),
 ('informatique', 171),
 ('agroalimentaire', 171),
 ('travail temporaire - intérim', 124),
 ('transports', 119),
 ('internet', 118),
 ('santé', 109),
 ('tourisme et loisirs', 93),
 ('enseignement supérieur - université', 88),
 ('banques', 87),
 ('bâtiment et construction', 83),
 ('services', 70),
 ('management', 66),
 ('premium', 58),
 ('télécommunications', 58)]

In [ ]:
categories = {
    "administration": ["administrations", "organismes", "ong"],
    "finance": ["finances", "banques", "microfinance", "transferts"],
    "commerce": ["commerce", "vente"],
    "rh": ["ressources humaines"],
    "education": ["formation", "éducation", "université", "écoles"],
    "tech": ["informatique", "internet", "ingénierie"],
    "communication": ["communication", "publicité", "marketing", "médias"],
    "sante": ["santé", "médical"],
    "transport": ["transport", "transit"],
    "agriculture": ["agroalimentaire", "agriculture"],
    "industrie": ["industrie", "mines", "pétrole"],
    "construction": ["bâtiment", "construction"],
    "tourisme": ["tourisme", "loisirs"],
    "immobilier": ["immobilier"],
    "energie": ["energie", "électrique"]
}

In [ ]:
def categorize_secteur(secteurs):
    found = []

    for cat, keywords in categories.items():
        for kw in keywords:
            for s in secteurs:
                if kw in s:
                    found.append(cat)
                    break

    return list(set(found))

In [ ]:
df["sector_category_final"] = df["secteur_list"].apply(categorize_secteur)

In [ ]:
df["sector_category_str"] = df["sector_category_final"].apply(lambda x: ", ".join(x))

In [ ]:
categories = {
    "public": [
        "administrations",
        "organisations non-gouvernementales (ong)",
        "organismes internationaux"
    ],

    "finance": [
        "finances",
        "banques",
        "microfinance",
        "transferts de fonds"
    ],

    "commerce": [
        "commerces",
        "vente en ligne"
    ],

    "rh": [
        "gestion des ressources humaines",
        "emploi",
        "agences de recrutement",
        "travail temporaire - intérim"
    ],

    "education": [
        "formations",
        "éducation",
        "enseignement supérieur - université"
    ],

    "tech": [
        "informatique",
        "internet",
        "télécommunications"
    ],

    "communication": [
        "communication",
        "publicité",
        "marketing & marketing digital",
        "médias"
    ],

    "sante": [
        "santé"
    ],

    "transport": [
        "transports"
    ],

    "agriculture": [
        "agroalimentaire"
    ],

    "industrie": [
        "industries",
        "mines - exploitations",
        "exploitation pétrolière"
    ],

    "construction": [
        "bâtiment et construction"
    ],

    "tourisme": [
        "tourisme et loisirs"
    ],

    "services": [
        "services"
    ],

    "management": [
        "management"
    ]
}

In [ ]:
def categorize_secteur(secteurs):
    found = []

    for cat, keywords in categories.items():
        for kw in keywords:
            for s in secteurs:
                if kw in s:
                    found.append(cat)
                    break

    return list(set(found))

In [ ]:
df["sector_category_final"] = df["secteur_list"].apply(categorize_secteur)

In [ ]:
df["sector_category_str"] = df["sector_category_final"].apply(lambda x: ", ".join(x))

In [ ]:
print(df.columns)

Index(['titre', 'ville', 'date_publication', 'lien', 'source', 'contrat',
       'entreprise', 'secteur', 'secteur_categorie', 'mois', 'Expérience',
       'description', 'exp_clean', 'experience_final', 'desc_clean',
       'desc_clean2', 'skills_raw', 'skills_final', 'dev_skills',
       'job_category', 'job_category_str', 'secteur_clean',
       'sector_category_final', 'sector_category_str', 'secteur_list'],
      dtype='object')


In [ ]:
df.head(10)

,titre,ville,date_publication,lien,source,contrat,entreprise,secteur,secteur_categorie,mois,Expérience,description,exp_clean,experience_final,desc_clean,desc_clean2,skills_raw,skills_final,dev_skills,job_category,job_category_str,secteur_clean,sector_category_final,sector_category_str,secteur_list
0,Agents distributeur,zone rurale,2026-04-10,https://senjob.com/sn/jobseekers/agents-distributeur_e_161306.html,senjob.com,NaN,NaN,NaN,Non spécifié,2026-04,None,"Description de l'offre A propos de nous : SamaMbey SUARL est une entreprise sociale évoluant dans le domaine agricole.Nous intervenons dans de nombreuses régions du Sénégal. Sama Mbey vise à améliorer les conditions de vie des petits producteurs en milieu rural et veut les aider à sortir de la pauvreté. Chez SamaMbey les petits agriculteurs peuvent se financer s'ils ont le bon outil pour payer petit à petit en avance pour les intrants. Notre vision et objectif à travers notre Étoile polaire sont de travailler avec 1 million de petits producteurs à l’horizon 2026. En plus, SamaMbey Suarl s’engage dans le domaine du développement communautaire en menant des actions de sensibilisation en nutrition, santé et hygiène. Intitulé du poste : Agent distributeur Localisation : La prestation se déroulera en milieu rural, dans les zones d'intervention de Sama Mbey, réparties sur trois régions administratives du Sénégal : Matam Bakel, Hamady Ounare, Matam, Thilogne Kolda Dabo, Bourocco,Kounkane, Médina Yoro Foula, Tanaff, Vélingara Sédhiou / Ziguinchor Bignona, Bounkiling, Diattacounda, Marsassoum, Médina Wandifa Nombre de postes: 36 Type de contrat: Prestation de service Rôles et responsabilités Les agents distributeurs auront pour missions : 1. VENTE & MOBILISATION - Développement commercial - Assister l'équipe de vente dans l'atteinte des objectifs de paquets finis - Réviser l'outil « Constitution de paquet 2026 » pour le maîtriser - Organiser les intrants selon l'outil de décharge",None,None,description de loffre a propos de nous samambey suarl est une entreprise sociale voluant dans le domaine agricolenous intervenons dans de nombreuses rgions du sngal sama mbey vise amliorer les conditions de vie des petits producteurs en milieu rural et veut les aider sortir de la pauvret chez samambey les petits agriculteurs peuvent se financer sils ont le bon outil pour payer petit petit en avance pour les intrants notre vision et objectif travers notre toile polaire sont de travailler avec 1 million de petits producteurs lhorizon 2026 en plus samambey suarl sengage dans le domaine du dveloppement communautaire en menant des actions de sensibilisation en nutrition sant et hygine intitul du poste agent distributeur localisation la prestation se droulera en milieu rural dans les zones dintervention de sama mbey rparties sur trois rgions administratives du sngal matam bakel hamady ounare matam thilogne kolda dabo bouroccokounkane mdina yoro foula tanaff vlingara sdhiou ziguinchor bignona bounkiling diattacounda marsassoum mdina wandifa nombre de postes 36 type de contrat prestation de service rles et responsabilits les agents distributeurs auront pour missions 1 vente mobilisation dveloppement commercial assister lquipe de vente dans latteinte des objectifs de paquets finis rviser loutil constitution de paquet 2026 pour le matriser organiser les intrants selon loutil de dcharge,de loffre a propos de nous samambey suarl est une entreprise sociale voluant dans le domaine agricolenous intervenons dans de nombreuses rgions du sngal sama mbey vise amliorer les conditions de vie des petits producteurs en milieu rural et veut les aider sortir de la pauvret chez samambey les petits agriculteurs peuvent se financer sils ont le bon outil pour payer petit petit en avance pour les intrants notre vision et objectif travers notre toile polaire sont de travailler avec 1 million de petits producteurs lhorizon 2026 en plus samambey suarl sengage dans le domaine du dveloppement communautaire en menant des actions de 

In [ ]:
df[[
    "titre",
    "secteur",
    "secteur_list",
    "sector_category_final",
    "skills_final",
    "Expérience",
    "experience_final"
]].head(10)

,titre,secteur,secteur_list,sector_category_final,skills_final,Expérience,experience_final
0,Agents distributeur,NaN,[],[],[],None,None
1,Stagiaire Rayonniste,NaN,[],[],[],None,None
2,Caissière,NaN,[],[],[],None,None
3,Stagiaire IT,NaN,[],[],[],None,None
4,Managing Director Everllence Senegal,NaN,[],[],[],None,None
5,Commissioning Engineer,NaN,[],[],[],None,None
6,HR Admin (H/F),NaN,[],[],[],None,None
7,Conseillers commerciaux confirmés et polyvalents,NaN,[],[],[],None,None
8,Conseillers en Assurance Santé,NaN,[],[],[],None,None
9,People Operations Specialist (Chargé RH),NaN,[],[],[],None,None


In [ ]:
cols_to_keep = [
    "titre",
    "ville",
    "date_publication",
    "contrat",
    "entreprise",
    "secteur",  # brut
    "sector_category_str",  # catégories propres
    "skills_str",  # compétences finales (si ok)
    "experience_final",
    "exp_min",
    "exp_max"
]

In [ ]:
df_clean = df[cols_to_keep].copy()

KeyError: "['skills_str', 'exp_min', 'exp_max'] not in index"

In [ ]:
df_clean.head(10)

,titre,ville,date_publication,contrat,entreprise,secteur,sector_category_str,skills_str,experience_final,exp_min,exp_max
0,Agents distributeur,zone rurale,2026-04-10,NaN,NaN,NaN,,,None,NaN,NaN
1,Stagiaire Rayonniste,Dakar,2026-04-09,NaN,NaN,NaN,,,None,NaN,NaN
2,Caissière,Dakar,2026-04-09,NaN,NaN,NaN,,,None,NaN,NaN
3,Stagiaire IT,Dakar,2026-04-09,NaN,NaN,NaN,,"html, comptabilité",None,NaN,NaN
4,Managing Director Everllence Senegal,Dakar,2026-04-09,NaN,NaN,NaN,,,None,NaN,NaN
5,Commissioning Engineer,Dakar,2026-04-09,NaN,NaN,NaN,,,None,NaN,NaN
6,HR Admin (H/F),Dakar,2026-04-09,NaN,NaN,NaN,,css,None,NaN,NaN
7,Conseillers commerciaux confirmés et polyvalents,Dakar,2026-04-09,NaN,NaN,NaN,,"communication, html, comptabilité",None,NaN,NaN
8,Conseillers en Assurance Santé,Dakar,2026-04-09,NaN,NaN,NaN,,html,None,NaN,NaN
9,People Operations Specialist (Chargé RH),Dakar,2026-04-09,NaN,NaN,NaN,,,None,NaN,NaN


In [ ]:
df_clean = df_clean.reset_index(drop=True)

In [ ]:
print(df.iloc[0])

titre                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [ ]:
df_final = df[[
    "titre",
    "ville",
    "date_publication",
    "source",
    "contrat",
    "entreprise",
    "secteur", # Original raw sector
    "secteur_categorie", # Original category
    "secteur_final_str", # New, cleaned sector
    "experience_final", # New, cleaned experience
    "exp_min",
    "exp_max",
    "skills_clean_final", # New, cleaned skills
    "mois"
]].copy()

KeyError: "['secteur_final_str', 'exp_min', 'exp_max', 'skills_clean_final'] not in index"

In [ ]:
df_final.head(10)

,intitule_poste,ville,date_publication,mois,contrat,entreprise,source,secteur,competences,experience,exp_min,exp_max
0,Agents distributeur,zone rurale,2026-04-10,2026-04-01,Non spécifié,Non spécifié,senjob.com,Non spécifié,"vente, relation commerciale, finance, santé, informatique",Non spécifié,NaN,NaN
1,Stagiaire Rayonniste,Dakar,2026-04-09,2026-04-01,Non spécifié,Non spécifié,senjob.com,Non spécifié,"informatique, vente, formation",Non spécifié,NaN,NaN
2,Caissière,Dakar,2026-04-09,2026-04-01,Non spécifié,Non spécifié,senjob.com,Non spécifié,informatique,Non spécifié,NaN,NaN
3,Stagiaire IT,Dakar,2026-04-09,2026-04-01,Non spécifié,Non spécifié,senjob.com,Non spécifié,"comptabilité, recrutement, finance, formation, informatique",Non spécifié,NaN,NaN
4,Managing Director Everllence Senegal,Dakar,2026-04-09,2026-04-01,Non spécifié,Non spécifié,senjob.com,Non spécifié,informatique,Non spécifié,NaN,NaN
5,Commissioning Engineer,Dakar,2026-04-09,2026-04-01,Non spécifié,Non spécifié,senjob.com,Non spécifié,informatique,Non spécifié,NaN,NaN
6,HR Admin (H/F),Dakar,2026-04-09,2026-04-01,Non spécifié,Non spécifié,senjob.com,Non spécifié,"informatique, recrutement, ressources humaines",Non spécifié,NaN,NaN
7,Conseillers commerciaux confirmés et polyvalents,Dakar,2026-04-09,2026-04-01,Non spécifié,Non spécifié,senjob.com,Non spécifié,"informatique, formation, communication, comptabilité",Non spécifié,NaN,NaN
8,Conseillers en Assurance Santé,Dakar,2026-04-09,2026-04-01,Non spécifié,Non spécifié,senjob.com,Non spécifié,"vente, relation commerciale, finance, formation, santé, informatique",Non spécifié,NaN,NaN
9,People Operations Specialist (Chargé RH),Dakar,2026-04-09,2026-04-01,Non spécifié,Non spécifié,senjob.com,Non spécifié,"finance, formation, banque, ressources humaines, santé, informatique",Non spécifié,NaN,NaN


In [ ]:
secteurs_ref = [
    "administrations",
    "finance", "banques",
    "communication", "publicité",
    "informatique", "internet",
    "agroalimentaire",
    "transports",
    "santé",
    "tourisme",
    "bâtiment",
    "éducation",
    "formation",
    "commerce",
    "vente",
    "ressources humaines",
    "industrie",
    "télécommunications"
]

In [ ]:
def extract_secteur_from_desc(text):
    found = []

    if pd.isna(text):
        return []

    text = text.lower()

    for secteur in secteurs_ref:
        if secteur in text:
            found.append(secteur)

    return list(set(found))

In [ ]:
df["secteur_from_desc"] = df.apply(
    lambda row: extract_secteur_from_desc(row["desc_clean2"])
    if pd.isna(row["secteur"]) or row["secteur"] == ""
    else [],
    axis=1
)

In [ ]:
def merge_secteur(row):
    if pd.isna(row["secteur"]) or row["secteur"] == "":
        return ", ".join(row["secteur_from_desc"])
    return row["secteur"]

df["secteur_final"] = df.apply(merge_secteur, axis=1)

In [ ]:
df[["secteur", "secteur_from_desc", "secteur_final"]].head(20)

,secteur,secteur_from_desc,secteur_final
0,NaN,"[vente, finance]","vente, finance"
1,NaN,"[vente, formation]","vente, formation"
2,NaN,[],
3,NaN,"[formation, finance]","formation, finance"
4,NaN,[industrie],industrie
5,NaN,[industrie],industrie
6,NaN,[ressources humaines],ressources humaines
7,NaN,"[communication, formation]","communication, formation"
8,NaN,"[formation, vente, finance]","formation, vente, finance"
9,NaN,"[formation, finance, banques]","formation, finance, banques"


In [ ]:
priority_order = [
    "finance",
    "banques",
    "informatique",
    "industrie",
    "santé",
    "télécommunications",
    "transport",
    "agroalimentaire",
    "communication",
    "ressources humaines",
    "commerce"
]

In [ ]:
def clean_secteur_list(secteurs):
    if not secteurs:
        return []

    # garder seulement ceux dans priorité
    filtered = [s for s in secteurs if s in priority_order]

    # trier selon priorité
    filtered.sort(key=lambda x: priority_order.index(x))

    # garder max 2
    return filtered[:2]

In [ ]:
df["secteur_final_clean"] = df["secteur_from_desc"].apply(clean_secteur_list)

In [ ]:
df["secteur_final_str"] = df["secteur_final_clean"].apply(lambda x: ", ".join(x))

In [ ]:
df[["secteur_from_desc", "secteur_final_clean"]].head(10)

,secteur_from_desc,secteur_final_clean
0,"[vente, finance]",[finance]
1,"[vente, formation]",[]
2,[],[]
3,"[formation, finance]",[finance]
4,[industrie],[industrie]
5,[industrie],[industrie]
6,[ressources humaines],[ressources humaines]
7,"[communication, formation]",[communication]
8,"[formation, vente, finance]",[finance]
9,"[formation, finance, banques]","[finance, banques]"


In [ ]:
mapping = {
    "banques": "finance",
    "microfinance": "finance",
    "transferts": "finance"
}

In [ ]:
import math

def normalize_secteur(secteurs):
    # Handle NaN values explicitly
    if isinstance(secteurs, float) and math.isnan(secteurs):
        return []

    # Ensure it's a list before proceeding, otherwise convert to empty list
    if not isinstance(secteurs, list):
        return []

    normalized = []
    for s in secteurs:
        if s in mapping:
            normalized.append(mapping[s])
        else:
            normalized.append(s)

    return list(set(normalized))

In [ ]:
df["secteur_final_clean"] = df["secteur_final_clean"].apply(normalize_secteur)

In [ ]:
df["secteur_final_clean"].explode().value_counts()

,count
secteur_final_clean,
finance,81
communication,56
informatique,10
industrie,9
commerce,8
ressources humaines,7
agroalimentaire,2


In [ ]:
df[df["secteur_final_clean"].apply(len) == 0].shape

(4065, 29)

In [ ]:
df["secteur_final_clean"].apply(lambda x: sorted(x)).head(10)

,secteur_final_clean
0,[finance]
1,[]
2,[]
3,[finance]
4,[industrie]
5,[industrie]
6,[ressources humaines]
7,[communication]
8,[finance]
9,[finance]


In [ ]:
df["secteur_final_clean"].explode().value_counts()

,count
secteur_final_clean,
finance,81
communication,56
informatique,10
industrie,9
commerce,8
ressources humaines,7
agroalimentaire,2


In [ ]:
def merge_and_stringfy_skills(row):
    all_skills = []
    if isinstance(row["skills_final"], list):
        all_skills.extend(row["skills_final"])
    if isinstance(row["dev_skills"], list):
        all_skills.extend(row["dev_skills"])
    return ", ".join(sorted(list(set(all_skills))))

df["skills_clean_final"] = df.apply(merge_and_stringfy_skills, axis=1)

In [ ]:
df[[
    "titre",
    "ville",
    "entreprise",
    "secteur_final_str",
    "skills_clean_final",
    "experience_final",
    "contrat"
]].head(20)

,titre,ville,entreprise,secteur_final_str,skills_clean_final,experience_final,contrat
0,Agents distributeur,zone rurale,NaN,finance,,None,NaN
1,Stagiaire Rayonniste,Dakar,NaN,,,None,NaN
2,Caissière,Dakar,NaN,,,None,NaN
3,Stagiaire IT,Dakar,NaN,finance,,None,NaN
4,Managing Director Everllence Senegal,Dakar,NaN,industrie,,None,NaN
5,Commissioning Engineer,Dakar,NaN,industrie,,None,NaN
6,HR Admin (H/F),Dakar,NaN,ressources humaines,,None,NaN
7,Conseillers commerciaux confirmés et polyvalents,Dakar,NaN,communication,,None,NaN
8,Conseillers en Assurance Santé,Dakar,NaN,finance,,None,NaN
9,People Operations Specialist (Chargé RH),Dakar,NaN,"finance, banques",,None,NaN


In [ ]:
df_final = df[[
    "titre",
    "ville",
    "date_publication",
    "contrat",
    "entreprise",
    "secteur_final_str",
    "skills_clean_final",
    "experience_final",
    "exp_min",
    "exp_max"
]].copy()

KeyError: "['exp_min', 'exp_max'] not in index"

In [ ]:
df_final = df_final.fillna("Non spécifié")

In [ ]:
df_final["competences"] = df_final["competences"].replace("", "Non spécifié")

In [ ]:
df_final.head(10)

,intitule_poste,ville,date_publication,mois,contrat,entreprise,source,secteur,competences,experience,exp_min,exp_max
0,Agents distributeur,zone rurale,2026-04-10 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"vente, relation commerciale, finance, santé, informatique",Non spécifié,Non spécifié,Non spécifié
1,Stagiaire Rayonniste,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"informatique, vente, formation",Non spécifié,Non spécifié,Non spécifié
2,Caissière,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,informatique,Non spécifié,Non spécifié,Non spécifié
3,Stagiaire IT,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"comptabilité, recrutement, finance, formation, informatique",Non spécifié,Non spécifié,Non spécifié
4,Managing Director Everllence Senegal,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,informatique,Non spécifié,Non spécifié,Non spécifié
5,Commissioning Engineer,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,informatique,Non spécifié,Non spécifié,Non spécifié
6,HR Admin (H/F),Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"informatique, recrutement, ressources humaines",Non spécifié,Non spécifié,Non spécifié
7,Conseillers commerciaux confirmés et polyvalents,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"informatique, formation, communication, comptabilité",Non spécifié,Non spécifié,Non spécifié
8,Conseillers en Assurance Santé,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"vente, relation commerciale, finance, formation, santé, informatique",Non spécifié,Non spécifié,Non spécifié
9,People Operations Specialist (Chargé RH),Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"finance, formation, banque, ressources humaines, santé, informatique",Non spécifié,Non spécifié,Non spécifié


In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].str.replace("banques", "finance")

KeyError: 'secteur_clean'

In [ ]:
df_final.head(10)

,intitule_poste,ville,date_publication,mois,contrat,entreprise,source,secteur,competences,experience,exp_min,exp_max
0,Agents distributeur,zone rurale,2026-04-10 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"vente, relation commerciale, finance, santé, informatique",Non spécifié,Non spécifié,Non spécifié
1,Stagiaire Rayonniste,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"informatique, vente, formation",Non spécifié,Non spécifié,Non spécifié
2,Caissière,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,informatique,Non spécifié,Non spécifié,Non spécifié
3,Stagiaire IT,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"comptabilité, recrutement, finance, formation, informatique",Non spécifié,Non spécifié,Non spécifié
4,Managing Director Everllence Senegal,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,informatique,Non spécifié,Non spécifié,Non spécifié
5,Commissioning Engineer,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,informatique,Non spécifié,Non spécifié,Non spécifié
6,HR Admin (H/F),Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"informatique, recrutement, ressources humaines",Non spécifié,Non spécifié,Non spécifié
7,Conseillers commerciaux confirmés et polyvalents,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"informatique, formation, communication, comptabilité",Non spécifié,Non spécifié,Non spécifié
8,Conseillers en Assurance Santé,Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"vente, relation commerciale, finance, formation, santé, informatique",Non spécifié,Non spécifié,Non spécifié
9,People Operations Specialist (Chargé RH),Dakar,2026-04-09 00:00:00,2026-04-01 00:00:00,Non spécifié,Non spécifié,senjob.com,Non spécifié,"finance, formation, banque, ressources humaines, santé, informatique",Non spécifié,Non spécifié,Non spécifié


In [ ]:
cols_text = [
    "contrat",
    "entreprise",
    "secteur_final_str",
    "skills_clean_final",
    "experience_final"
]

df_final[cols_text] = df_final[cols_text].fillna("Non spécifié")

KeyError: "['secteur_final_str', 'skills_clean_final', 'experience_final'] not in index"

In [ ]:
df_final[["exp_min", "exp_max"]].head()

,exp_min,exp_max
0,Non spécifié,Non spécifié
1,Non spécifié,Non spécifié
2,Non spécifié,Non spécifié
3,Non spécifié,Non spécifié
4,Non spécifié,Non spécifié


In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].apply(
    lambda x: ", ".join(set(x.split(", "))) if x != "Non spécifié" else x
)

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].replace("", "Non spécifié")

In [ ]:
df_final.info()

In [ ]:
df_final = df_final.rename(columns={
    "skills_clean_final": "competences",
    "secteur_final_str": "secteur_clean", # Renaming the new, cleaned sector to avoid confusion
    "experience_final": "experience"
})

In [ ]:
df_final = df_final.rename(columns={
    "titre": "intitule_poste"
})

In [ ]:
df_final.to_csv("dataset_final.csv", index=False)

In [ ]:
from google.colab import files
files.download("dataset_final.csv")

In [ ]:
# Removed: This dropped the cleaned sector column unintentionally

In [ ]:
print(df_final.columns)

In [ ]:
df_final.isna().sum()

In [ ]:
df_final.head()

In [ ]:
# Removed: secteur_categorie is already included from the initial selection

In [ ]:
# Removed: source is already included from the initial selection

In [ ]:
# Removed: mois is already included from the initial selection

In [ ]:
def extract_skills(text):
    if pd.isna(text):
        return []

    text = text.lower()
    skills = []

    # DATA
    if "donnée" in text:
        if "analyse" in text:
            skills.append("analyse de données")
        if "collecte" in text:
            skills.append("collecte de données")
        if "traitement" in text:
            skills.append("traitement de données")
        if "visualisation" in text:
            skills.append("visualisation de données")

    # TECH / IT
    if "développement" in text:
        if "application" in text:
            skills.append("développement d'applications")
        if "api" in text:
            skills.append("développement d'api")
        if "web" in text:
            skills.append("développement web")
    if "informatique" in text or "it" in text:
        skills.append("informatique")

    # MARKETING / COMMUNICATION
    if "marketing digital" in text:
        skills.append("marketing digital")
    elif "marketing" in text:
        skills.append("marketing")
    if "communication" in text:
        skills.append("communication")
    if "réseaux sociaux" in text:
        skills.append("gestion des réseaux sociaux")

    # FINANCE
    if "finance" in text or "financier" in text:
        skills.append("finance")
    if "comptabilité" in text:
        skills.append("comptabilité")
    if "banque" in text:
        skills.append("banque")

    # RH
    if "ressources humaines" in text or "rh" in text:
        skills.append("ressources humaines")
    if "recrutement" in text:
        skills.append("recrutement")

    # PROJET
    if "gestion de projet" in text:
        skills.append("gestion de projet")
    elif "projet" in text:
        skills.append("gestion de projet")

    # COMMERCE / VENTE
    if "vente" in text:
        skills.append("vente")
    if "commercial" in text:
        skills.append("relation commerciale")

    # LOGISTIQUE
    if "logistique" in text:
        skills.append("logistique")
    if "transport" in text:
        skills.append("transport")

    # SANTE
    if "santé" in text or "médical" in text:
        skills.append("santé")

    # EDUCATION
    if "formation" in text:
        skills.append("formation")

    return list(set(skills))

In [ ]:
df["skills_final"] = df["desc_clean2"].apply(extract_skills)

In [ ]:
df["skills_clean_final"] = df["skills_final"].apply(lambda x: ", ".join(x) if x else "Non spécifié")

In [ ]:
df_final[["intitule_poste", "competences"]].head(100)

In [ ]:
# Re-apply skill extraction (ensures skills_clean_final is updated in df)
df["skills_final"] = df["desc_clean2"].apply(extract_skills)
df["skills_clean_final"] = df["skills_final"].apply(lambda x: ", ".join(x) if x else "Non spécifié")

# Re-construct df_final with all intended columns from df
df_final = df[[
    "titre",
    "ville",
    "date_publication",
    "source",
    "contrat",
    "entreprise",
    "secteur", # Original raw sector
    "secteur_categorie", # Original category
    "secteur_final_str", # New, cleaned sector
    "experience_final", # New, cleaned experience
    "exp_min",
    "exp_max",
    "skills_clean_final", # New, cleaned skills
    "mois"
]].copy()

# Fill NaN values with "Non spécifié"
df_final = df_final.fillna("Non spécifié")

# Rename columns in one go
df_final = df_final.rename(columns={
    "titre": "intitule_poste",
    "skills_clean_final": "competences",
    "secteur_final_str": "secteur_clean",
    "experience_final": "experience"
})

# Apply final string cleaning operations now that columns are renamed
df_final["competences"] = df_final["competences"].replace("", "Non spécifié")
df_final["secteur_clean"] = df_final["secteur_clean"].str.replace("banques", "finance")
df_final["secteur_clean"] = df_final["secteur_clean"].apply(
    lambda x: ", ".join(set(x.split(", "))) if x != "Non spécifié" else x
)
df_final["secteur_clean"] = df_final["secteur_clean"].replace("", "Non spécifié")

# Display the head of the final DataFrame
df_final.head(10)

In [ ]:
df["skills_final"] = df["desc_clean2"].apply(extract_skills)

In [ ]:
df["skills_clean_final"] = df["skills_final"].apply(lambda x: ", ".join(x) if x else "Non spécifié")

In [ ]:
df_final = df[[
    "titre",
    "ville",
    "date_publication",
    "source",
    "contrat",
    "entreprise",
    "secteur", # Original raw sector
    "secteur_categorie", # Original category
    "secteur_final_str", # New, cleaned sector
    "experience_final", # New, cleaned experience
    "exp_min",
    "exp_max",
    "skills_clean_final", # New, cleaned skills
    "mois"
]].copy()

In [ ]:
df_final = df_final.fillna("Non spécifié")

In [ ]:
df_final = df_final.rename(columns={
    "skills_clean_final": "competences",
    "secteur_final_str": "secteur_clean",
    "experience_final": "experience"
})

In [ ]:
df_final = df_final.rename(columns={
    "titre": "intitule_poste"
})

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].str.replace("banques", "finance")

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].apply(
    lambda x: ", ".join(set(x.split(", "))) if x != "Non spécifié" else x
)

In [ ]:
df_final.head(10)

In [ ]:
df["skills_final"] = df["desc_clean2"].apply(extract_skills)

In [ ]:
df["skills_clean_final"] = df["skills_final"].apply(lambda x: ", ".join(x) if x else "Non spécifié")

In [ ]:
df_final = df[[
    "titre",
    "ville",
    "date_publication",
    "source",
    "contrat",
    "entreprise",
    "secteur", # Original raw sector
    "secteur_categorie", # Original category
    "secteur_final_str", # New, cleaned sector
    "experience_final", # New, cleaned experience
    "exp_min",
    "exp_max",
    "skills_clean_final", # New, cleaned skills
    "mois"
]].copy()

In [ ]:
df_final = df_final.fillna("Non spécifié")

In [ ]:
df_final = df_final.rename(columns={
    "skills_clean_final": "competences",
    "secteur_final_str": "secteur_clean",
    "experience_final": "experience"
})

In [ ]:
df_final = df_final.rename(columns={
    "titre": "intitule_poste"
})

In [ ]:
df_final["competences"] = df_final["competences"].replace("", "Non spécifié")

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].str.replace("banques", "finance")

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].apply(
    lambda x: ", ".join(set(x.split(", "))) if x != "Non spécifié" else x
)

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].replace("", "Non spécifié")

In [ ]:
df_final.head(10)

In [ ]:
df["skills_final"] = df["desc_clean2"].apply(extract_skills)

In [ ]:
df["skills_clean_final"] = df["skills_final"].apply(lambda x: ", ".join(x) if x else "Non spécifié")

In [ ]:
df_final = df[[
    "titre",
    "ville",
    "date_publication",
    "source",
    "contrat",
    "entreprise",
    "secteur", # Original raw sector
    "secteur_categorie", # Original category
    "secteur_final_str", # New, cleaned sector
    "experience_final", # New, cleaned experience
    "exp_min",
    "exp_max",
    "skills_clean_final", # New, cleaned skills
    "mois"
]].copy()

In [ ]:
df_final = df_final.fillna("Non spécifié")

In [ ]:
df_final["competences"] = df_final["competences"].replace("", "Non spécifié")

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].str.replace("banques", "finance")

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].apply(
    lambda x: ", ".join(set(x.split(", "))) if x != "Non spécifié" else x
)

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].replace("", "Non spécifié")

In [ ]:
df_final = df_final.rename(columns={
    "skills_clean_final": "competences",
    "secteur_final_str": "secteur_clean", # Renaming the new, cleaned sector to avoid confusion
    "experience_final": "experience"
})

In [ ]:
df_final = df_final.rename(columns={
    "titre": "intitule_poste"
})

In [ ]:
df_final.head(100)

In [ ]:
# Re-apply skill extraction (ensures skills_clean_final is updated in df)
df["skills_final"] = df["desc_clean2"].apply(extract_skills)
df["skills_clean_final"] = df["skills_final"].apply(lambda x: ", ".join(x) if x else "Non spécifié")

# Re-construct df_final with all intended columns from df
df_final = df[[
    "titre",
    "ville",
    "date_publication",
    "source",
    "contrat",
    "entreprise",
    "secteur", # Original raw sector
    "secteur_categorie", # Original category
    "secteur_final_str", # New, cleaned sector
    "experience_final", # New, cleaned experience
    "exp_min",
    "exp_max",
    "skills_clean_final", # New, cleaned skills
    "mois"
]].copy()

# Fill NaN values with "Non spécifié"
df_final = df_final.fillna("Non spécifié")

# Rename columns in one go
df_final = df_final.rename(columns={
    "titre": "intitule_poste",
    "skills_clean_final": "competences",
    "secteur_final_str": "secteur_clean",
    "experience_final": "experience"
})

# Apply final string cleaning operations now that columns are renamed
df_final["competences"] = df_final["competences"].replace("", "Non spécifié")
df_final["secteur_clean"] = df_final["secteur_clean"].str.replace("banques", "finance")
df_final["secteur_clean"] = df_final["secteur_clean"].apply(
    lambda x: ", ".join(set(x.split(", "))) if x != "Non spécifié" else x
)
df_final["secteur_clean"] = df_final["secteur_clean"].replace("", "Non spécifié")

# Display the head of the final DataFrame
df_final.head(10)

In [ ]:
df["skills_final"] = df["desc_clean2"].apply(extract_skills)

In [ ]:
df["skills_clean_final"] = df["skills_final"].apply(lambda x: ", ".join(x) if x else "Non spécifié")

In [ ]:
df_final = df[[
    "titre",
    "ville",
    "date_publication",
    "source",
    "contrat",
    "entreprise",
    "secteur", # Original raw sector
    "secteur_categorie", # Original category
    "secteur_final_str", # New, cleaned sector
    "experience_final", # New, cleaned experience
    "exp_min",
    "exp_max",
    "skills_clean_final", # New, cleaned skills
    "mois"
]].copy()

In [ ]:
df_final = df_final.fillna("Non spécifié")

In [ ]:
df_final = df_final.rename(columns={
    "skills_clean_final": "competences",
    "secteur_final_str": "secteur_clean",
    "experience_final": "experience"
})

In [ ]:
df_final = df_final.rename(columns={
    "titre": "intitule_poste"
})

In [ ]:
df_final["competences"] = df_final["competences"].replace("", "Non spécifié")

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].str.replace("banques", "finance")

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].apply(
    lambda x: ", ".join(set(x.split(", "))) if x != "Non spécifié" else x
)

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].replace("", "Non spécifié")

In [ ]:
df_final.head(10)

In [ ]:
df["skills_final"] = df["desc_clean2"].apply(extract_skills)

In [ ]:
df["skills_clean_final"] = df["skills_final"].apply(lambda x: ", ".join(x) if x else "Non spécifié")

In [ ]:
df_final = df[[
    "titre",
    "ville",
    "date_publication",
    "source",
    "contrat",
    "entreprise",
    "secteur", # Original raw sector
    "secteur_categorie", # Original category
    "secteur_final_str", # New, cleaned sector
    "experience_final", # New, cleaned experience
    "exp_min",
    "exp_max",
    "skills_clean_final", # New, cleaned skills
    "mois"
]].copy()

In [ ]:
df_final = df_final.fillna("Non spécifié")

In [ ]:
df_final = df_final.rename(columns={
    "skills_clean_final": "competences",
    "secteur_final_str": "secteur_clean",
    "experience_final": "experience"
})

In [ ]:
df_final = df_final.rename(columns={
    "titre": "intitule_poste"
})

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].str.replace("banques", "finance")

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].apply(
    lambda x: ", ".join(set(x.split(", "))) if x != "Non spécifié" else x
)

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].replace("", "Non spécifié")

In [ ]:
df_final.head(10)

In [ ]:
df["skills_final"] = df["desc_clean2"].apply(extract_skills)

In [ ]:
df["skills_clean_final"] = df["skills_final"].apply(lambda x: ", ".join(x) if x else "Non spécifié")

In [ ]:
df_final = df[[
    "titre",
    "ville",
    "date_publication",
    "source",
    "contrat",
    "entreprise",
    "secteur", # Original raw sector
    "secteur_categorie", # Original category
    "secteur_final_str", # New, cleaned sector
    "experience_final", # New, cleaned experience
    "exp_min",
    "exp_max",
    "skills_clean_final", # New, cleaned skills
    "mois"
]].copy()

In [ ]:
df_final = df_final.fillna("Non spécifié")

In [ ]:
df_final = df_final.rename(columns={
    "skills_clean_final": "competences",
    "secteur_final_str": "secteur_clean",
    "experience_final": "experience"
})

In [ ]:
df_final = df_final.rename(columns={
    "titre": "intitule_poste"
})

In [ ]:
df_final["competences"] = df_final["competences"].replace("", "Non spécifié")

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].str.replace("banques", "finance")

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].apply(
    lambda x: ", ".join(set(x.split(", "))) if x != "Non spécifié" else x
)

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].replace("", "Non spécifié")

In [ ]:
df_final.head(10)

In [ ]:
df["skills_final"] = df["desc_clean2"].apply(extract_skills)

In [ ]:
df["skills_clean_final"] = df["skills_final"].apply(lambda x: ", ".join(x) if x else "Non spécifié")

In [ ]:
df_final = df[[
    "titre",
    "ville",
    "date_publication",
    "source",
    "contrat",
    "entreprise",
    "secteur", # Original raw sector
    "secteur_categorie", # Original category
    "secteur_final_str", # New, cleaned sector
    "experience_final", # New, cleaned experience
    "exp_min",
    "exp_max",
    "skills_clean_final", # New, cleaned skills
    "mois"
]].copy()

In [ ]:
df_final = df_final.fillna("Non spécifié")

In [ ]:
df_final = df_final.rename(columns={
    "skills_clean_final": "competences",
    "secteur_final_str": "secteur_clean",
    "experience_final": "experience"
})

In [ ]:
df_final = df_final.rename(columns={
    "titre": "intitule_poste"
})

In [ ]:
df_final["competences"] = df_final["competences"].replace("", "Non spécifié")

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].str.replace("banques", "finance")

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].apply(
    lambda x: ", ".join(set(x.split(", "))) if x != "Non spécifié" else x
)

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].replace("", "Non spécifié")

In [ ]:
df_final.head(10)

In [ ]:
df_final["secteur_clean"] = df_final["secteur_clean"].replace("", "Non spécifié")

In [ ]:
df = df.rename(columns={"titre": "intitule_poste"})

In [ ]:
df[["intitule_poste", "skills_clean_final"]].head(15)

In [ ]:
# Afficher toutes les colonnes sans troncature
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

# Afficher un aperçu complet
print("===== APERÇU DU DATASET =====")
display(df_final.head(10))

print("\n===== LISTE DES COLONNES =====")
print(df_final.columns)

print("\n===== TYPES DE DONNÉES =====")
print(df.info())

print("\n===== VALEURS MANQUANTES =====")
print(df_final.isna().sum())

In [ ]:
display(df.head(5))
display(df_final.head(5))

In [ ]:
df_final = df_final.rename(columns={
    "skills_clean_final": "competences",
    "experience_final": "experience"
})

In [ ]:
df_final.head()

In [ ]:
df_final.head()

In [ ]:
def final_secteur(row):
    if row["secteur_categorie"] != "Non spécifié":
        return row["secteur_categorie"]
    elif row["secteur_clean"] != "" and pd.notna(row["secteur_clean"]):
        return row["secteur_clean"]
    else:
        return "Non spécifié"

df_final["secteur"] = df.apply(final_secteur, axis=1)

In [ ]:
df_final = df_final.drop(columns=[
    "secteur_categorie",
    "secteur_clean"
])

In [ ]:
df_final[["intitule_poste", "secteur"]].head(100)

In [ ]:
df_final = df_final.drop(columns=[
    "secteur",
    "secteur_categorie",
    "secteur_clean"
], errors="ignore")

In [ ]:
df_final.head()

In [ ]:
df_final["secteur"] = df.apply(
    lambda row: row["secteur_categorie"]
    if row["secteur_categorie"] != "Non spécifié"
    else row["secteur_clean"],
    axis=1
)

In [ ]:
df_final["secteur"] = df_final["secteur"].replace("", "Non spécifié")

In [ ]:
df_final[["intitule_poste", "secteur"]].head(10)

In [ ]:
df_final = df_final.drop(columns=[
    "secteur_categorie",
    "secteur_clean"
], errors="ignore")

In [ ]:
df_final.head(10)

In [ ]:
df["secteur_final"] = df.apply(
    lambda row: row["secteur_categorie"]
    if row["secteur_categorie"] != "Non spécifié"
    else row["secteur_clean"],
    axis=1
)

In [ ]:
df["secteur_final"] = df["secteur_final"].replace("", "Non spécifié")

In [ ]:
cols_final = [
    "intitule_poste",
    "ville",
    "date_publication",
    "source",
    "contrat",
    "entreprise",
    "secteur_final",
    "skills_clean_final",
    "experience_final",
    "exp_min",
    "exp_max",
    "mois"
]

df_final = df[cols_final].copy()

In [ ]:
df_final = df_final.rename(columns={
    "skills_clean_final": "competences",
    "experience_final": "experience",
    "secteur_final": "secteur"
})

In [ ]:
df_final.head()

In [ ]:
df[["secteur_categorie", "secteur_clean", "secteur_final"]].head(500)

In [ ]:
df["secteur_final"] = df["secteur_final"].replace("premium", "Non spécifié")

In [ ]:
cols_final = [
    "intitule_poste",
    "ville",
    "date_publication",
    "mois",
    "contrat",
    "entreprise",
    "source",
    "secteur_final",
    "skills_clean_final",
    "experience_final",
    "exp_min",
    "exp_max"
]

df_final = df[cols_final].copy()

# Renommer pour version métier
df_final = df_final.rename(columns={
    "secteur_final": "secteur",
    "skills_clean_final": "competences",
    "experience_final": "experience"
})

In [ ]:
cols_text = [
    "contrat",
    "entreprise",
    "secteur",
    "competences",
    "experience"
]

df_final[cols_text] = df_final[cols_text].fillna("Non spécifié")

df_final["competences"] = df_final["competences"].replace("", "Non spécifié")
df_final["secteur"] = df_final["secteur"].replace("", "Non spécifié")

In [ ]:
df_final["ville"] = df_final["ville"].fillna("Non spécifié")

In [ ]:
df_final["mois"] = pd.to_datetime(df_final["mois"], format="%Y-%m", errors="coerce")

In [ ]:
df_final["date_publication"] = pd.to_datetime(df_final["date_publication"], errors="coerce")

In [ ]:
df_final.head()
df_final.isna().sum()

In [ ]:
df_final.info()

In [ ]:
df_final.duplicated().sum()

In [ ]:
df_final.to_csv("dataset_final.csv", index=False)
from google.colab import files
files.download("dataset_final.csv")

In [ ]:
import re

def extract_min_max(exp):
    if pd.isna(exp):
        return None, None

    exp = str(exp)

    numbers = re.findall(r"\d+", exp)

    if len(numbers) == 1:
        val = int(numbers[0])
        return val, val

    elif len(numbers) >= 2:
        return int(numbers[0]), int(numbers[1])

    return None, None


df[["exp_min", "exp_max"]] = df["experience"].apply(
    lambda x: pd.Series(extract_min_max(x))
)